# 🧠 AI-Based Neural Network System for Function Approximation
## Derivative Computation and Tangent Line Estimation Using Automatic Differentiation

---

**Student:** Mujeeb-Ur-Rehman Sahito 
**Roll No:** 25-BSCS-43 
**University:** Sheikh Ayaz University 
**Department:** Computer Science Department

---

## Table of Contents

1. [Introduction](#1-introduction)
2. [Setup and Imports](#2-setup)
3. [Dataset Generation](#3-dataset)
4. [Neural Network Architecture](#4-architecture)
5. [Training](#5-training)
6. [Derivative Estimation](#6-derivatives)
7. [Tangent Line Computation](#7-tangent)
8. [Performance Metrics](#8-metrics)
9. [Visualization Gallery](#9-visualization)
10. [Implicit Function (Circle)](#10-circle)
11. [PINN Comparison](#11-pinn)
12. [Discussion and Conclusion](#12-conclusion)

## 1. Introduction <a id='1-introduction'></a>

This notebook demonstrates an AI-based system that uses **deep neural networks** to:

- **Approximate mathematical functions** using feedforward neural networks (PyTorch)
- **Compute derivatives** using automatic differentiation (autograd) and finite differences
- **Estimate tangent lines** at specified points and compare with true analytic solutions
- **Handle implicit functions** (conic sections) with implicit differentiation
- **Evaluate performance** using comprehensive metrics (MSE, MAE, RMSE, R²)

### Functions Studied

| Function | Expression | Domain |
|----------|-----------|--------|
| Explicit | $f(x) = x^3 - 2x^2 + \sin(x)$ | $[-3, 3]$ |
| Implicit | $x^2 + y^2 = 25$ | Circle, r=5 |

## 2. Setup and Imports <a id='2-setup'></a>

In [ ]:
# Core imports
import sys
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, os.path.dirname(os.getcwd()) if 'notebooks' in os.getcwd() else os.getcwd())

# Project imports
from src.dataset import (
    set_seed, generate_dataset, create_dataloaders,
    explicit_function, explicit_derivative,
    explicit_derivative_torch, generate_both_datasets
)
from src.model import FunctionApproximator, PINNApproximator, CircleApproximator, get_device
from src.trainer import Trainer, PINNTrainer
from src.derivatives import AutoDiffCalculator, FiniteDifferenceCalculator, compute_all_derivatives
from src.tangent import TangentLineCalculator
from src.metrics import PerformanceMetrics
from src.visualization import Visualizer
from src.implicit_curve import CircleCurve, CircleTrainer, run_circle_experiment

# Setup
set_seed(42)
device = get_device()
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {device}')
print(f'NumPy version: {np.__version__}')

# Professional plot style
%matplotlib inline
plt.rcParams.update({'figure.figsize': (12, 7), 'figure.dpi': 100, 'font.size': 12})
sns.set_context('notebook', font_scale=1.1)

## 3. Dataset Generation <a id='3-dataset'></a>

We generate a synthetic dataset from the explicit function:

$$f(x) = x^3 - 2x^2 + \sin(x), \quad x \in [-3, 3]$$

Both **clean** (noise-free) and **noisy** (Gaussian noise) datasets are created.

In [ ]:
# Generate both clean and noisy datasets
datasets = generate_both_datasets(n_samples=1000, noise_std=0.05, seed=42)

x_clean, y_clean = datasets['clean']
x_noisy, y_noisy = datasets['noisy']

print(f'Clean dataset: x.shape={x_clean.shape}, y.shape={y_clean.shape}')
print(f'Noisy dataset: x.shape={x_noisy.shape}, y.shape={y_noisy.shape}')
print(f'x range: [{x_clean.min():.2f}, {x_clean.max():.2f}]')
print(f'y range (clean): [{y_clean.min():.2f}, {y_clean.max():.2f}]')

In [ ]:
# Visualize datasets
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

ax1.plot(x_clean, y_clean, '#1976D2', linewidth=2.5)
ax1.set_title('Clean Dataset: f(x) = x³ − 2x² + sin(x)', fontsize=14, fontweight='bold')
ax1.set_xlabel('x'); ax1.set_ylabel('f(x)'); ax1.grid(alpha=0.3)

ax2.scatter(x_noisy, y_noisy, c='#E53935', alpha=0.4, s=10, label='Noisy data')
ax2.plot(x_clean, y_clean, '#1976D2', linewidth=2, label='True function')
ax2.set_title('Noisy Dataset (σ = 0.05)', fontsize=14, fontweight='bold')
ax2.set_xlabel('x'); ax2.set_ylabel('f(x)'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Create DataLoaders
train_loader, val_loader = create_dataloaders(x_clean, y_clean, batch_size=32, seed=42)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

## 4. Neural Network Architecture <a id='4-architecture'></a>

We use a **feedforward neural network** with the following architecture:

```
Input(1) → Dense(128, ReLU) → Dense(128, ReLU) → Dense(64, ReLU) → Dense(32, ReLU) → Output(1)
```

- **Xavier/Glorot** weight initialization
- **Adam** optimizer with weight decay
- **ReduceLROnPlateau** learning rate scheduler
- **Early stopping** (patience=50)

In [ ]:
# Create the model
set_seed(42)
model = FunctionApproximator(hidden_layers=[128, 128, 64, 32], activation='relu')
print(model.get_architecture_summary())
print(f'\nModel structure:\n{model}')

## 5. Training <a id='5-training'></a>

Training the neural network with **1000 epochs**, **batch size 32**, and **MSE loss**.

In [ ]:
# Train the model
trainer = Trainer(
    model=model, learning_rate=1e-3, weight_decay=1e-5,
    device=device, scheduler_type='reduce_on_plateau',
    early_stopping_patience=50
)

history = trainer.fit(train_loader, val_loader, epochs=1000, verbose=True, log_interval=200)

print(f'\nTraining complete!')
print(f'Best validation loss: {min(history["val_loss"]):.6e}')
print(f'Total epochs: {len(history["train_loss"])}')

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

epochs = range(1, len(history['train_loss']) + 1)
ax1.semilogy(epochs, history['train_loss'], '#1976D2', label='Train')
ax1.semilogy(epochs, history['val_loss'], '#E53935', label='Validation')
ax1.set_title('Training Loss (Log Scale)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.semilogy(epochs, history['learning_rate'], '#8E24AA')
ax2.set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('LR'); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Get predictions
model.eval()
with torch.no_grad():
    x_tensor = torch.FloatTensor(x_clean.reshape(-1, 1)).to(device)
    y_pred = model(x_tensor).cpu().numpy().flatten()

# Plot function approximation
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

ax1.plot(x_clean, y_clean, '#1976D2', linewidth=2.5, label=r'True: $f(x) = x^3 - 2x^2 + \sin(x)$')
ax1.plot(x_clean, y_pred, '#E53935', linewidth=2, linestyle='--', label='Neural Network')
ax1.fill_between(x_clean, y_clean, y_pred, alpha=0.1, color='red')
ax1.set_title('Function Approximation', fontsize=16, fontweight='bold')
ax1.set_ylabel('f(x)'); ax1.legend(fontsize=12); ax1.grid(alpha=0.3)

residual = y_clean - y_pred
ax2.fill_between(x_clean, residual, 0, alpha=0.3, color='#43A047')
ax2.plot(x_clean, residual, '#43A047', linewidth=1.5)
ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_xlabel('x'); ax2.set_ylabel('Residual'); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f'MSE: {np.mean(residual**2):.6e}')

## 6. Derivative Estimation <a id='6-derivatives'></a>

We compute derivatives using two methods:

1. **Automatic Differentiation** (PyTorch autograd) — exact for the neural network
2. **Finite Difference** (central difference) — $f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$

True derivative: $f'(x) = 3x^2 - 4x + \cos(x)$

In [ ]:
# Compute all derivatives
deriv_results = compute_all_derivatives(model, x_clean, explicit_function, explicit_derivative, device)

true_d = deriv_results['true_derivative']
ad_d = deriv_results['autograd_derivative']
fd_d = deriv_results['finite_diff_derivative']

# Plot comparison
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

ax1.plot(x_clean, true_d, '#263238', linewidth=2.5, label=r"True: $f'(x) = 3x^2 - 4x + \cos(x)$")
ax1.plot(x_clean, ad_d, '#1976D2', linewidth=2, linestyle='--', label='AutoDiff (PyTorch)')
ax1.plot(x_clean, fd_d, '#E53935', linewidth=2, linestyle=':', label='Finite Difference')
ax1.set_title('Derivative Comparison', fontsize=16, fontweight='bold')
ax1.set_ylabel("f'(x)"); ax1.legend(fontsize=11); ax1.grid(alpha=0.3)

ax2.semilogy(x_clean, np.abs(true_d - ad_d) + 1e-15, '#1976D2', label='AutoDiff error')
ax2.semilogy(x_clean, np.abs(true_d - fd_d) + 1e-15, '#E53935', label='FiniteDiff error')
ax2.set_xlabel('x'); ax2.set_ylabel('|Error|'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'AutoDiff MSE vs True: {np.mean((true_d - ad_d)**2):.6e}')
print(f'FiniteDiff MSE vs True: {np.mean((true_d - fd_d)**2):.6e}')

## 7. Tangent Line Computation <a id='7-tangent'></a>

At $x_0 = 1.5$, we compute:
- **True tangent line** using the analytic derivative
- **AI tangent line** using the neural network + autograd

Tangent equation: $y = f'(x_0)(x - x_0) + f(x_0)$

In [ ]:
# Compute tangent lines at x0 = 1.5
x0 = 1.5
x_range = np.linspace(x0 - 2, x0 + 2, 200).astype(np.float32)

true_tangent = TangentLineCalculator.compute_true_tangent(x0, explicit_function, explicit_derivative, x_range)
ai_tangent = TangentLineCalculator.compute_ai_tangent(x0, model, device, x_range)
comparison = TangentLineCalculator.compare_tangent_lines(true_tangent, ai_tangent)

print(f'True tangent: {true_tangent["equation_point_form"]}')
print(f'AI tangent:   {ai_tangent["equation_point_form"]}')
print(f'\nSlope error:     {comparison["slope_error"]:.6e}')
print(f'Intercept error: {comparison["intercept_error"]:.6e}')

In [ ]:
# Plot tangent comparison
fig, ax = plt.subplots(figsize=(13, 8))

ax.plot(x_clean, y_clean, '#263238', linewidth=2.5, label=r'$f(x)$')
ax.plot(x_range, true_tangent['y_tangent'], '#1976D2', linewidth=2.5, label=f"True: {true_tangent['equation']}")
ax.plot(x_range, ai_tangent['y_tangent'], '#E53935', linewidth=2.5, linestyle='--', label=f"AI: {ai_tangent['equation']}")
ax.scatter([x0], [true_tangent['f_x0']], color='#FB8C00', s=150, zorder=5, edgecolors='black', linewidths=1.5, label=f'x₀ = {x0}')

ax.set_title('Tangent Line: True vs AI-Derived', fontsize=16, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Performance Metrics <a id='8-metrics'></a>

Comprehensive evaluation of the system's performance.

In [ ]:
# Compute all metrics
metrics = PerformanceMetrics.compute_all_metrics(
    y_true=y_clean, y_pred=y_pred,
    true_tangent=true_tangent, ai_tangent=ai_tangent,
    true_deriv=true_d, ai_deriv=ad_d
)

# Display metrics table
table = PerformanceMetrics.format_metrics_table(metrics, 'Complete Performance Metrics')
print(table)

# Tangent comparison table
comp_table = PerformanceMetrics.format_comparison_table(true_tangent, ai_tangent, comparison)
print(comp_table)

## 9. Visualization Gallery <a id='9-visualization'></a>

In [ ]:
# Error analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

func_err = y_clean - y_pred
deriv_err = true_d - ad_d

axes[0,0].hist(func_err, bins=40, color='#1976D2', alpha=0.7, edgecolor='white')
axes[0,0].axvline(0, color='black', linestyle='--')
axes[0,0].set_title('Function Error Distribution', fontweight='bold')

axes[0,1].hist(deriv_err, bins=40, color='#E53935', alpha=0.7, edgecolor='white')
axes[0,1].axvline(0, color='black', linestyle='--')
axes[0,1].set_title('Derivative Error Distribution', fontweight='bold')

axes[1,0].scatter(x_clean, np.abs(func_err), c='#1976D2', alpha=0.5, s=15)
axes[1,0].set_yscale('log'); axes[1,0].set_title('Function |Error| vs x', fontweight='bold')
axes[1,0].set_xlabel('x')

axes[1,1].scatter(x_clean, np.abs(deriv_err), c='#E53935', alpha=0.5, s=15)
axes[1,1].set_yscale('log'); axes[1,1].set_title('Derivative |Error| vs x', fontweight='bold')
axes[1,1].set_xlabel('x')

for ax in axes.flat: ax.grid(alpha=0.3)
fig.suptitle('Error Analysis', fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 10. Implicit Function: Circle <a id='10-circle'></a>

We study the implicit circle:

$$x^2 + y^2 = 25 \quad (r = 5)$$

Implicit differentiation gives:

$$\frac{dy}{dx} = -\frac{x}{y}$$

We train a neural network on the upper semicircle and compute tangent at $x_0 = 3$.

In [ ]:
# Run circle experiment
circle_results = run_circle_experiment(radius=5.0, x0=3.0, n_samples=500, epochs=500, device=device)

print(f'\nCircle Experiment Results:')
print(f'  True y0: {circle_results["y0"]:.6f}')
print(f'  AI y0:   {circle_results["ai_tangent"]["y0"]:.6f}')
print(f'  True slope: {circle_results["true_tangent"]["slope"]:.6f}')
print(f'  AI slope:   {circle_results["ai_tangent"]["slope"]:.6f}')
print(f'  MSE: {circle_results["metrics"]["function_mse"]:.6e}')
print(f'  R²:  {circle_results["metrics"]["function_r_squared"]:.6f}')

In [ ]:
# Plot circle
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Full circle with tangent
theta = np.linspace(0, 2*np.pi, 500)
ax1.plot(5*np.cos(theta), 5*np.sin(theta), '#263238', linewidth=2.5, label='x² + y² = 25')
ax1.plot(circle_results['x_plot'], circle_results['y_pred'], '#1976D2', linewidth=2, linestyle='--', label='NN Approximation')

ct = circle_results['true_tangent']
ca = circle_results['ai_tangent']
if ct.get('y_tangent') is not None:
    ax1.plot(ct['x_range'], ct['y_tangent'], '#43A047', linewidth=2.5, label='True Tangent')
if ca.get('y_tangent') is not None:
    ax1.plot(ca['x_range'], ca['y_tangent'], '#E53935', linewidth=2.5, linestyle='--', label='AI Tangent')

ax1.scatter([3], [circle_results['y0']], color='#FB8C00', s=150, zorder=5, edgecolors='black', linewidths=1.5)
ax1.set_aspect('equal'); ax1.set_title('Circle with Tangent Lines', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10); ax1.grid(alpha=0.3); ax1.set_xlim(-7, 7); ax1.set_ylim(-7, 7)

# Training loss
ch = circle_results['history']
ax2.semilogy(ch['train_loss'], '#1976D2', label='Train')
ax2.semilogy(ch['val_loss'], '#E53935', label='Val')
ax2.set_title('Circle Model Training', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 11. PINN Comparison <a id='11-pinn'></a>

We compare the standard neural network with a **Physics-Informed Neural Network (PINN)** that includes the derivative constraint in the loss function.

$$\mathcal{L}_{\text{PINN}} = \mathcal{L}_{\text{data}} + \lambda \cdot \mathcal{L}_{\text{physics}}$$

where $\mathcal{L}_{\text{physics}} = \text{MSE}(\hat{f}'(x), f'(x))$

In [ ]:
# Train PINN
set_seed(42)
model_pinn = PINNApproximator(hidden_layers=[128, 128, 64, 32], activation='tanh')
pinn_trainer = PINNTrainer(
    model=model_pinn, true_derivative_func=explicit_derivative_torch,
    physics_weight=0.1, learning_rate=1e-3, device=device,
    early_stopping_patience=50
)
history_pinn = pinn_trainer.fit(train_loader, val_loader, epochs=1000, verbose=True, log_interval=200)

# Get PINN predictions
model_pinn.eval()
with torch.no_grad():
    y_pred_pinn = model_pinn(x_tensor).cpu().numpy().flatten()

print(f'\nStandard NN MSE: {np.mean((y_clean - y_pred)**2):.6e}')
print(f'PINN MSE:        {np.mean((y_clean - y_pred_pinn)**2):.6e}')

In [ ]:
# Compare NN vs PINN
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].plot(x_clean, y_clean, '#263238', linewidth=2.5, label='True')
axes[0].plot(x_clean, y_pred, '#1976D2', linewidth=2, linestyle='--', label='Standard NN')
axes[0].plot(x_clean, y_pred_pinn, '#43A047', linewidth=2, linestyle=':', label='PINN')
axes[0].set_title('Function Approximation', fontweight='bold'); axes[0].legend()

axes[1].semilogy(x_clean, np.abs(y_clean - y_pred)+1e-15, '#1976D2', label='NN |Error|')
axes[1].semilogy(x_clean, np.abs(y_clean - y_pred_pinn)+1e-15, '#43A047', label='PINN |Error|')
axes[1].set_title('Error Comparison', fontweight='bold'); axes[1].legend()

axes[2].semilogy(history['val_loss'], '#1976D2', label='NN Val Loss')
axes[2].semilogy(history_pinn['val_loss'], '#43A047', label='PINN Val Loss')
axes[2].set_title('Validation Loss', fontweight='bold'); axes[2].legend()

for ax in axes: ax.grid(alpha=0.3); ax.set_xlabel('x' if ax != axes[2] else 'Epoch')
fig.suptitle('Standard NN vs Physics-Informed NN', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 12. Discussion and Conclusion <a id='12-conclusion'></a>

### Key Findings

1. **Function Approximation**: The feedforward neural network successfully approximated $f(x) = x^3 - 2x^2 + \sin(x)$ with very low MSE and R² ≈ 1.0.

2. **Derivative Computation**: Automatic differentiation (PyTorch autograd) provides exact derivatives of the neural network, closely matching the true analytic derivative.

3. **Tangent Line Estimation**: The AI-derived tangent line closely matches the true tangent line, with minimal slope and intercept errors.

4. **Implicit Functions**: The neural network successfully learned the upper semicircle mapping, and implicit tangent lines were accurately computed.

5. **PINN**: Adding the physics-informed loss term (derivative constraint) can improve the network's understanding of the underlying function structure.

### Strengths
- Neural networks provide differentiable function approximations
- Automatic differentiation enables exact derivative computation for the NN
- The system generalizes to both explicit and implicit functions
- PINN incorporates domain knowledge into training

### Limitations
- Neural networks are approximate; residual errors exist
- Performance depends on architecture and hyperparameter tuning
- Training requires computational resources
- Extrapolation beyond the training domain is unreliable

---

**End of Notebook** 
Mujeeb-Ur-Rehman Sahito (25-BSCS-43) · Sheikh Ayaz University